# 01 Data Quality Assessment

This notebook performs a full data quality assessment for `data/CardiacPatientData.csv` without modifying the raw data file.

The assessment includes dataset structure, missingness, duplicates, repeated IDs, descriptive statistics, categorical/binary value distributions, clinical range checks, IQR outlier checks, and basic visualizations saved to `reports/figures/`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "CardiacPatientData.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
SUMMARY_PATH = REPORTS_DIR / "data_quality_summary.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Summary CSV path: {SUMMARY_PATH}")

## Load data

The raw CSV is read into memory only. This notebook writes derived summary outputs to `reports/` and figures to `reports/figures/`; it does **not** modify or overwrite the raw CSV.

In [ ]:
df = pd.read_csv(DATA_PATH)
# Preserve the original in-memory dataframe for auditing; all derived objects are copies/summaries.
df_raw = df.copy(deep=True)
df.head()

## 1. Dataset shape

In [ ]:
shape_summary = pd.DataFrame({"metric": ["rows", "columns"], "value": [df.shape[0], df.shape[1]]})
display(shape_summary)

## 2. Column names and data types

In [ ]:
dtype_summary = (
    df.dtypes.astype(str)
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(dtype_summary)

## 3. Missing value counts and percentages

In [ ]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
})
display(missing_summary.sort_values(["missing_count", "column"], ascending=[False, True]))

## 4. Duplicate row count

In [ ]:
duplicate_row_count = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_row_count:,}")

duplicate_examples = df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(20)
if duplicate_row_count:
    display(duplicate_examples)
else:
    print("No duplicate rows found.")

## 5. Repeated ID analysis

In [ ]:
id_column = "ID" if "ID" in df.columns else None

if id_column:
    id_counts = df[id_column].value_counts(dropna=False)
    repeated_id_counts = id_counts[id_counts > 1]
    repeated_id_summary = pd.DataFrame({
        "metric": [
            "total_rows",
            "unique_ids_including_missing",
            "ids_repeated_more_than_once",
            "rows_with_repeated_ids",
            "max_records_per_id",
        ],
        "value": [
            len(df),
            df[id_column].nunique(dropna=False),
            len(repeated_id_counts),
            int(repeated_id_counts.sum()),
            int(id_counts.max()) if len(id_counts) else 0,
        ],
    })
    display(repeated_id_summary)
    display(
        repeated_id_counts.rename_axis(id_column)
        .reset_index(name="row_count")
        .sort_values("row_count", ascending=False)
        .head(25)
    )
else:
    repeated_id_counts = pd.Series(dtype=int)
    repeated_id_summary = pd.DataFrame()
    print("No ID column found.")

## 6. Summary statistics for numeric columns

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_summary = df[numeric_columns].describe().T
numeric_summary["missing_count"] = df[numeric_columns].isna().sum()
numeric_summary["missing_percent"] = (df[numeric_columns].isna().mean() * 100).round(2)
display(numeric_summary)

## 7. Value counts for categorical/binary columns

In [ ]:
# Treat object/category/bool columns and low-cardinality numeric columns as categorical/binary for distribution review.
low_cardinality_threshold = 15
categorical_columns = [
    col for col in df.columns
    if (
        not pd.api.types.is_numeric_dtype(df[col])
        or df[col].nunique(dropna=True) <= low_cardinality_threshold
        or col in ["Gender", "Alcoholic", "Smoke", "FHCD", "TriageScore", "Outcome"]
    )
]

value_count_tables = {}
for col in categorical_columns:
    counts = df[col].value_counts(dropna=False).rename("count")
    percents = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2).rename("percent")
    table = pd.concat([counts, percents], axis=1).reset_index().rename(columns={"index": col})
    value_count_tables[col] = table
    print(f"\n{col}")
    display(table)

## 8. Invalid or impossible physiological values

The ranges below are intentionally broad clinical plausibility ranges intended for data-quality screening, not diagnosis. `BT` is handled dynamically: if the median temperature is above 60, the notebook assumes Fahrenheit; otherwise it assumes Celsius.

> Note: the dataset column name is `Ceratinine`; this notebook preserves the source spelling and checks it as creatinine.

In [ ]:
# Broad plausible ranges for data-quality screening.
clinical_ranges = {
    "Age": {"min": 0, "max": 120, "unit": "years"},
    "SBP": {"min": 50, "max": 260, "unit": "mmHg"},
    "DBP": {"min": 30, "max": 160, "unit": "mmHg"},
    "HR": {"min": 20, "max": 250, "unit": "beats/min"},
    "RR": {"min": 5, "max": 60, "unit": "breaths/min"},
    "SpO2": {"min": 0, "max": 100, "unit": "%"},
    "GCS": {"min": 3, "max": 15, "unit": "score"},
    "TriageScore": {"min": 1, "max": 5, "unit": "score"},
    "Na": {"min": 110, "max": 180, "unit": "mmol/L"},
    "K": {"min": 1.5, "max": 9.0, "unit": "mmol/L"},
    "Cl": {"min": 70, "max": 140, "unit": "mmol/L"},
    "Urea": {"min": 0, "max": 250, "unit": "mg/dL or local lab unit"},
    "Ceratinine": {"min": 0, "max": 1500, "unit": "µmol/L or local lab unit"},
}

if "BT" in df.columns:
    bt_median = pd.to_numeric(df["BT"], errors="coerce").median()
    if pd.notna(bt_median) and bt_median > 60:
        clinical_ranges["BT"] = {"min": 86, "max": 113, "unit": "°F", "assumption": "Fahrenheit inferred from median > 60"}
    else:
        clinical_ranges["BT"] = {"min": 30, "max": 45, "unit": "°C", "assumption": "Celsius inferred from median <= 60"}

range_rows = []
invalid_examples = {}
for col, rule in clinical_ranges.items():
    if col not in df.columns:
        continue
    values = pd.to_numeric(df[col], errors="coerce")
    invalid_mask = values.notna() & ((values < rule["min"]) | (values > rule["max"]))
    range_rows.append({
        "column": col,
        "min_allowed": rule["min"],
        "max_allowed": rule["max"],
        "unit": rule["unit"],
        "assumption": rule.get("assumption", ""),
        "invalid_count": int(invalid_mask.sum()),
        "invalid_percent": round(float(invalid_mask.mean() * 100), 2),
        "observed_min": values.min(),
        "observed_max": values.max(),
    })
    if invalid_mask.any():
        invalid_examples[col] = df.loc[invalid_mask, [c for c in ["ID", col] if c in df.columns]].head(20)

invalid_value_summary = pd.DataFrame(range_rows).sort_values("invalid_count", ascending=False)
display(invalid_value_summary)

for col, examples in invalid_examples.items():
    print(f"\nInvalid examples for {col}")
    display(examples)

## 9. Outlier checks for clinical variables

In [ ]:
clinical_variables = [
    "Age", "SBP", "DBP", "HR", "RR", "BT", "SpO2", "GCS", "TriageScore",
    "Na", "K", "Cl", "Urea", "Ceratinine",
]
clinical_variables = [col for col in clinical_variables if col in df.columns]

outlier_rows = []
outlier_examples = {}
for col in clinical_variables:
    values = pd.to_numeric(df[col], errors="coerce")
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask = values.notna() & ((values < lower) | (values > upper))
    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_fence": lower,
        "upper_fence": upper,
        "outlier_count": int(outlier_mask.sum()),
        "outlier_percent": round(float(outlier_mask.mean() * 100), 2),
    })
    if outlier_mask.any():
        outlier_examples[col] = df.loc[outlier_mask, [c for c in ["ID", col] if c in df.columns]].head(20)

outlier_summary = pd.DataFrame(outlier_rows).sort_values("outlier_count", ascending=False)
display(outlier_summary)

for col, examples in outlier_examples.items():
    print(f"\nIQR outlier examples for {col}")
    display(examples)

## 10. Basic visualizations saved to `reports/figures/`

In [ ]:
saved_figures = []

# Missingness bar chart
plt.figure(figsize=(10, 5))
missing_plot_data = missing_summary.sort_values("missing_percent", ascending=False)
sns.barplot(data=missing_plot_data, x="missing_percent", y="column", color="#4C78A8")
plt.title("Missing Values by Column")
plt.xlabel("Missing (%)")
plt.ylabel("Column")
plt.tight_layout()
path = FIGURES_DIR / "missing_values_by_column.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
saved_figures.append(path)
plt.show()

# Histograms for clinical numeric variables
hist_columns = [col for col in clinical_variables if pd.api.types.is_numeric_dtype(df[col])]
if hist_columns:
    ncols = 3
    nrows = int(np.ceil(len(hist_columns) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, hist_columns):
        sns.histplot(pd.to_numeric(df[col], errors="coerce").dropna(), kde=True, ax=ax, color="#59A14F")
        ax.set_title(f"Distribution of {col}")
    for ax in axes[len(hist_columns):]:
        ax.axis("off")
    fig.tight_layout()
    path = FIGURES_DIR / "clinical_numeric_distributions.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    saved_figures.append(path)
    plt.show()

# Boxplots for clinical numeric variables
if hist_columns:
    plot_df = df[hist_columns].apply(pd.to_numeric, errors="coerce").melt(var_name="variable", value_name="value")
    plt.figure(figsize=(max(12, len(hist_columns) * 0.9), 6))
    sns.boxplot(data=plot_df, x="variable", y="value", color="#F28E2B")
    plt.xticks(rotation=45, ha="right")
    plt.title("Clinical Variable Boxplots")
    plt.tight_layout()
    path = FIGURES_DIR / "clinical_variable_boxplots.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    saved_figures.append(path)
    plt.show()

# Outcome distribution, if present
if "Outcome" in df.columns:
    plt.figure(figsize=(6, 4))
    outcome_counts = df["Outcome"].value_counts(dropna=False).reset_index()
    outcome_counts.columns = ["Outcome", "count"]
    sns.barplot(data=outcome_counts, x="Outcome", y="count", color="#E15759")
    plt.title("Outcome Distribution")
    plt.xlabel("Outcome")
    plt.ylabel("Count")
    plt.tight_layout()
    path = FIGURES_DIR / "outcome_distribution.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    saved_figures.append(path)
    plt.show()

# Correlation heatmap for numeric variables
if len(numeric_columns) > 1:
    plt.figure(figsize=(12, 10))
    corr = df[numeric_columns].corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
    plt.title("Numeric Feature Correlation Heatmap")
    plt.tight_layout()
    path = FIGURES_DIR / "numeric_correlation_heatmap.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    saved_figures.append(path)
    plt.show()

print("Saved figures:")
for fig_path in saved_figures:
    print(f"- {fig_path.relative_to(PROJECT_ROOT)}")

## Save consolidated data quality summary CSV

In [ ]:
summary = pd.DataFrame({"column": df.columns})
summary["dtype"] = summary["column"].map(df.dtypes.astype(str).to_dict())
summary["missing_count"] = summary["column"].map(df.isna().sum().to_dict()).astype(int)
summary["missing_percent"] = summary["column"].map((df.isna().mean() * 100).round(2).to_dict())
summary["unique_count"] = summary["column"].map(df.nunique(dropna=False).to_dict()).astype(int)
summary["duplicate_row_count_dataset"] = duplicate_row_count
summary["is_clinical_range_checked"] = summary["column"].isin(invalid_value_summary["column"])

invalid_lookup = invalid_value_summary.set_index("column") if not invalid_value_summary.empty else pd.DataFrame()
outlier_lookup = outlier_summary.set_index("column") if not outlier_summary.empty else pd.DataFrame()

summary["invalid_count"] = summary["column"].map(invalid_lookup["invalid_count"].to_dict() if not invalid_lookup.empty else {}).fillna(0).astype(int)
summary["invalid_percent"] = summary["column"].map(invalid_lookup["invalid_percent"].to_dict() if not invalid_lookup.empty else {}).fillna(0.0)
summary["clinical_min_allowed"] = summary["column"].map(invalid_lookup["min_allowed"].to_dict() if not invalid_lookup.empty else {})
summary["clinical_max_allowed"] = summary["column"].map(invalid_lookup["max_allowed"].to_dict() if not invalid_lookup.empty else {})
summary["clinical_unit"] = summary["column"].map(invalid_lookup["unit"].to_dict() if not invalid_lookup.empty else {})
summary["outlier_count_iqr"] = summary["column"].map(outlier_lookup["outlier_count"].to_dict() if not outlier_lookup.empty else {}).fillna(0).astype(int)
summary["outlier_percent_iqr"] = summary["column"].map(outlier_lookup["outlier_percent"].to_dict() if not outlier_lookup.empty else {}).fillna(0.0)

if id_column:
    summary["repeated_id_count_dataset"] = len(repeated_id_counts)
    summary["rows_with_repeated_ids_dataset"] = int(repeated_id_counts.sum())
else:
    summary["repeated_id_count_dataset"] = np.nan
    summary["rows_with_repeated_ids_dataset"] = np.nan

summary.to_csv(SUMMARY_PATH, index=False)
display(summary)
print(f"Saved summary CSV to {SUMMARY_PATH.relative_to(PROJECT_ROOT)}")

## Raw data integrity check

In [ ]:
# Confirm this notebook did not mutate the in-memory raw dataframe copy.
pd.testing.assert_frame_equal(df, df_raw)
print("Raw CSV was not modified by this notebook; all outputs are derived summaries/figures.")